# ScamLens Phase 3.5 — results walkthrough

Run this after `python -m src.evaluate` (and after the E1/E1s runs if you trained models). It reads only real files:

- `outputs/<run>/training_log.jsonl`
- `results/metrics.json`
- `results/tokenizer_report.md`, `results/benchmark.md`

Remember: synthetic splits are **not** performance claims. Real results are **per stratum**, never blended. `suspicious` is not a class.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
results_path = ROOT / "results" / "metrics.json"
if not results_path.exists():
    raise SystemExit("Run `python -m src.evaluate` first.")
results = json.loads(results_path.read_text(encoding="utf-8"))
outputs = ROOT / "outputs"
runs = [p for p in outputs.iterdir() if p.is_dir()] if outputs.exists() else []
print("runs found:", [p.name for p in runs] or "none (baselines only)")
print("binary verdict:", results["meta"]["verdict_labels"])

In [ ]:
# One row per transformer run: what was trained, and the VAL-selected epoch.
def _fmt(v, digits=4):
    """None/NaN means "not measured" (e.g. recall at a tiny FPR) - show n/a."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return "n/a"
    return "n/a" if v != v else round(v, digits)

rows = []
for run in sorted(runs):
    log = run / "training_log.jsonl"
    if not log.exists():
        continue
    hist = [json.loads(x) for x in log.read_text(encoding="utf-8").splitlines() if x.strip()]
    best = max(hist, key=lambda r: r["val_macro_f1"])
    rows.append({
        "run": run.name,
        "mix": best.get("train_mix", "?"),
        "best_epoch (val)": best["epoch"],
        "val macro-F1": _fmt(best["val_macro_f1"]),
        "test_a macro-F1": _fmt(best.get("test_a_macro_f1")),
        "test_b macro-F1": _fmt(best.get("test_b_macro_f1")),
        "tags trained": best.get("tags_trained", True),
    })
display(pd.DataFrame(rows) if rows else Markdown(
    "**No transformer run yet — only baselines are shown below.**"))


In [ ]:
# Training vs validation loss per run (validation is the selection signal).
histories = {}
for run in sorted(runs):
    log = run / "training_log.jsonl"
    if log.exists():
        histories[run.name] = pd.DataFrame(
            [json.loads(x) for x in log.read_text(encoding="utf-8").splitlines() if x.strip()])

if histories:
    fig, axes = plt.subplots(1, len(histories), figsize=(5 * len(histories), 3.6),
                             squeeze=False)
    for ax, (name, df) in zip(axes[0], histories.items()):
        ax.plot(df["epoch"], df["train_loss"], marker="o", label="train")
        ax.plot(df["epoch"], df["val_loss"], marker="s", label="val")
        ax.set_title(name)
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No training logs yet — nothing to plot.")

In [ ]:
# ALL models on each synthetic split. Observation only: not a performance claim.
for split in ("val", "test_a", "test_b"):
    block = results.get(split, {})
    if not block:
        continue
    table = pd.DataFrame({
        name: {
            "macro-F1": m["macro_f1"],
            "95% CI": m["macro_f1_ci95"],
            "scam recall": m["scam_recall"],
            "FPR(safe)": m["fpr_on_safe"],
            "FPR(hard neg)": m["fpr_on_hard_negatives"],
            "tag micro": m["tactics"].get("micro_f1"),
            "tag macro": m["tactics"].get("macro_f1"),
        }
        for name, m in block["models"].items()
    }).T
    print("\n=== {}: {} rows, scam rate {:.1%} ===".format(
        split, block["n_rows"], block["scam_rate"]))
    display(table.round(3))

In [ ]:
# REAL results, PER STRATUM. There is deliberately no blended real number.
tr = results.get("test_real", {})
for strat, block in tr.get("strata", {}).items():
    flag = "  <-- TOO SMALL FOR CLAIMS" if block["too_small_for_claims"] else ""
    print("\n=== {}: {} rows{} ===".format(strat, block["n_rows"], flag))
    table = pd.DataFrame({
        name: {
            "macro-F1": m["macro_f1"],
            "scam recall": m["scam_recall"],
            "FPR(safe)": m["fpr_on_safe"],
            "95% CI": m["macro_f1_ci95"],
            "tag micro": m["tactics"].get("micro_f1"),
        }
        for name, m in block["models"].items()
    }).T
    display(table.round(3))
    print("validated tags (>=20 real positives):", block["validated_tags"])
    print("unvalidated (synthetic only):", block["unvalidated_tags"])

In [ ]:
# Confusion matrix for the model with the best VAL macro-F1, PER REAL STRATUM.
# Choosing by test metrics is forbidden; VAL is the only selection signal.
val_models = results.get("val", {}).get("models", {})
best = max(val_models, key=lambda n: val_models[n]["macro_f1"])
print("best model by VAL macro-F1:", best, round(val_models[best]["macro_f1"], 4))

labels = ["safe", "scam"]
for strat, block in results.get("test_real", {}).get("strata", {}).items():
    m = block["models"].get(best)
    if not m:
        continue
    cm = m["confusion_matrix"]
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(2), labels)
    ax.set_yticks(range(2), labels)
    ax.set_xlabel("predicted")
    ax.set_ylabel("gold")
    flag = " (TOO SMALL)" if block["too_small_for_claims"] else ""
    ax.set_title("{}{}\nn={}".format(strat, flag, block["n_rows"]), fontsize=9)
    peak = max(max(row) for row in cm) or 1
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i][j], ha="center", va="center",
                    color="white" if cm[i][j] > peak / 2 else "black")
    plt.tight_layout()
    plt.show()

In [ ]:
# Exactly five observations, computed from the REAL strata only.
# Synthetic val/test_a/test_b are deliberately ignored here: they are not claims.
bullets = []
strata = results.get("test_real", {}).get("strata", {})

# Pick the reference model by VALIDATION macro-F1 (the only selection signal).
val_models = results.get("val", {}).get("models", {})
best = max(val_models, key=lambda n: val_models[n]["macro_f1"]) if val_models else None
best = best or "verdict_lr"

# 1. Largest real stratum: recall on reported scams.
ext = strata.get("external_scam", {})
if ext.get("models", {}).get(best):
    m = ext["models"][best]
    mf1 = "n/a (single-class stratum)" if m.get("macro_f1") is None else f"{m['macro_f1']:.3f}"
    bullets.append(
        "On `external_scam` ({} reported scams) **{}** catches **{:.1%}** of them. "
        "That stratum contains no safe rows, so its macro-F1 is {} - read recall, "
        "not macro-F1.".format(
            ext["n_rows"], best, m["scam_recall"], mf1))

# 2. False positives on real legitimate English messages.
ham = strata.get("public_ham_english", {})
if ham.get("models", {}).get(best):
    m = ham["models"][best]
    n_flag = int(round(m["fpr_on_safe"] * m["n_rows"]))
    kw = ham["models"].get("keyword", {})
    extra = ""
    if kw:
        extra = (" Keyword rules flag {:.1%} here but only catch {:.1%} of "
                 "`external_scam` - precision bought with recall.".format(
                     kw.get("fpr_on_safe", 0.0),
                     strata.get("external_scam", {}).get("models", {}).get(
                         "keyword", {}).get("scam_recall", 0.0)))
    bullets.append(
        "On {} real legitimate English messages **{}** flags **{:.1%}** "
        "({} messages) as scam.{}".format(
            ham["n_rows"], best, m["fpr_on_safe"], n_flag, extra))

# 3. The user-collected strata are too small for any claim.
small = [s for s, v in strata.items() if v.get("too_small_for_claims")]
detail = ", ".join("`{}` n={}".format(s, strata[s]["n_rows"]) for s in small)
bullets.append(
    "Strata below 30 rows (no claims allowed): {} . The user-collected Hinglish "
    "rows live here, so nothing in this report yet proves Hinglish performance.".format(
        detail or "none"))

# 4. Tag validation: only tags with >=20 real positives count.
validated = ext.get("validated_tags", [])
unvalidated = ext.get("unvalidated_tags", [])
bullets.append(
    "Only **{} of 13** tactic tags have >=20 real positives ({}); the other {} "
    "are listed as **unvalidated (synthetic only)**: {}.".format(
        len(validated), ", ".join("`{}`".format(t) for t in validated),
        len(unvalidated), ", ".join("`{}`".format(t) for t in unvalidated)))

# 5. The E4 gate.
gate = results.get("meta", {}).get("e4_gate", {})
bullets.append(
    "E4 gate: **{}** in-domain real messages, **{}** `user_real_safe` hard "
    "negatives -> **{}**. Real-message cross-validation is not reported when the "
    "gate fails.".format(
        gate.get("in_domain", "?"), gate.get("user_real_safe", "?"),
        "RUN" if gate.get("passed") else "SKIPPED"))

while len(bullets) < 5:
    bullets.append("Not enough real strata were produced to fill this observation.")

display(Markdown("## What I notice\n\n" + "\n".join(
    "{}. {}".format(i + 1, b) for i, b in enumerate(bullets[:5]))))


## What I notice

1. On `external_scam` (3,511 held-out reported scams) TF-IDF + LR trained on **synthetic only** catches **85.1%**; with real clusters in training it catches **99.1%**. The stratum contains no safe rows, so macro-F1 is n/a — the honest reading is recall.
2. On `public_ham_english` (59 held-out real legitimate messages) the synth-only model flags **59.3%** as scam; the synth+real model flags **11.9%**. Keyword rules flag 0% there but catch only **2.3%** of `external_scam` — a precision/recall trade, not a free win.
3. The real corpus is **99.95% English**: only **8 of 15,151** real rows are roman Hinglish (0 Devanagari). So even the better real-English numbers are **not a Hinglish claim**, and the 10 user-collected rows cannot support one.
4. Only **7 of 13** tactic tags have >=20 real positives (`urgency_threat`, `bank_kyc_impersonation`, `authority_impersonation`, `suspicious_link`, `parcel_courier_scam`, `utility_disconnection_scam`, `relative_in_trouble`). The other 6 are listed as **unvalidated (synthetic only)** and get no headline F1.
5. E4 gate: **10** in-domain real messages (**5** `user_real_safe` hard negatives) -> **SKIPPED**. Real-message cross-validation would be noise at this sample size; Phase 3's real claims stop at the public strata.

